# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

## Результаты сохранённого прогона

Следующая ячейка пересчитывает метрики из сохранённых предсказаний. Запустите её из каталога `hw7`. Достаточно Python, установка ML-библиотек и GPU для этого шага не нужны.

Дальше идут ячейки для нового эксперимента: загрузка моделей, квантизация, генерация ответов, SFT и оценка на dev. По умолчанию они используют greedy decoding и сохраняют результаты в `artifacts_reproduced/`. Выводы в конце разделов описывают исходный прогон, в котором сработало наследование параметров sampling.


In [1]:
from pathlib import Path

from recompute_metrics import print_saved_metrics

print_saved_metrics(Path("artifacts_compression_hw"))

run	quality	intent_accuracy	slot_micro_f1	frame_exact_match	valid_json_rate	n
awq_massive	0.2163	0.2891	0.1436	0.0625	0.4141	256
baseline	0.2639	0.3398	0.1880	0.1094	0.4844	256
gptq_g64_massive	0.2884	0.3867	0.1900	0.0977	0.5977	256
gptq_massive	0.1654	0.2305	0.1003	0.0586	0.3555	256
gptq_wikitext	0.1654	0.2500	0.0809	0.0352	0.3359	256
smoothquant_05_massive	0.2717	0.3633	0.1800	0.1055	0.4883	256
smoothquant_08_massive	0.2625	0.3555	0.1696	0.0977	0.4648	256
student_128_lr1e-5	0.5170	0.7031	0.3310	0.2344	0.9805	256
student_large_lr1e-5	0.5916	0.8086	0.3746	0.3008	0.9883	256
student_sft	0.5916	0.8086	0.3746	0.3008	0.9883	256
teacher	0.5598	0.7656	0.3540	0.3047	0.9688	256


### Сперва настроим окружение и подготовим код для экспериментов

In [ ]:
# Выполните в новом окружении, затем перезапустите kernel:
%pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
    "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
    "numpy==2.2.6" "pandas==2.2.3" "safetensors==0.8.0" "huggingface-hub<1" tqdm

In [ ]:
import gc
import json
import os
import re
import shutil
from collections.abc import Sequence
from importlib.metadata import version
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)
from recompute_metrics import compute_metrics, parse_answer


In [ ]:
SEED = 42
ARTIFACTS = Path(os.environ.get("HW7_ARTIFACTS_DIR", "artifacts_reproduced"))
DEV_RESULTS = ARTIFACTS / "dev"
DEV_RESULTS.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

def require_saved_checkpoint(path: Path) -> None:
    """Проверяет наличие конфига, токенизатора и всех шардов весов."""
    index = path / "model.safetensors.index.json"
    if index.is_file():
        weight_map = json.loads(index.read_text(encoding="utf-8"))["weight_map"]
        files = set(weight_map.values())
    else:
        files = {"model.safetensors"}
    files.update({"config.json", "tokenizer_config.json"})
    for name in sorted(files):
        file = path / name
        if not file.is_file() or file.stat().st_size == 0:
            raise FileNotFoundError(f"Для возобновления не хватает файла {file}")


environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors",
    )
}
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [ ]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

In [ ]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [ ]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [ ]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})
train_data = data["train"]
DEV_SAMPLES = 256
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

In [ ]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. В новом запуске используем greedy decoding и одинаковый лимит токенов.

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen3-8B" # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = 32

# True — использовать чекпоинты от того же прогона из artifacts_compression_hw.
RESUME_QUANTIZATION = False
# False — greedy. None — наследовать настройки модели, как в исходном прогоне.
USE_MODEL_DEFAULTS = False

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """

    ### YOUR CODE HERE

    answer = []
    for token_id in token_ids:
        answer.append(token_id)
        if token_id == eos_token_id:
            return answer

    while answer and answer[-1] == pad_token_id:
        answer.pop()
    return answer

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    if not prompts:
        return []
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    ### YOUR CODE HERE
    ### Fill input_ids and attention_mask
    
    for ids in prompts:
        pad_count = width - len(ids)
        input_ids.append([config.pad_token_id] * pad_count + ids)
        attention_mask.append([0] * pad_count + [1] * len(ids))

    device = model.get_input_embeddings().weight.device
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long, device=device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config, use_model_defaults=USE_MODEL_DEFAULTS,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [ ]:
BASELINE_PATH = ARTIFACTS / "baseline"

baseline_model = None
if RESUME_QUANTIZATION:
    require_saved_checkpoint(BASELINE_PATH)
    tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)
else:
    ensure_empty_dir(BASELINE_PATH)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    baseline_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16,
        device_map=DEVICE,
    ).eval()
    baseline_model.save_pretrained(BASELINE_PATH, safe_serialization=True)
    tokenizer.save_pretrained(BASELINE_PATH)
print("Исходная модель:", BASE_MODEL_ID)
print("Пример prompt, токенов:", len(encode_prompt(tokenizer, train_data[0]["text"])))

## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [ ]:
def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    ### YOUR CODE HERE
    
    for start in tqdm(range(0, len(texts), GENERATE_BATCH_SIZE), desc="Оценка"):
        batch_texts = texts[start:start + GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, text) for text in batch_texts]
        answer_ids = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answer_ids, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset, *, device: str = DEVICE,
) -> tuple[dict[str, float], list[str]]:
    """Загружает сохранённый checkpoint, оценивает его и выгружает из памяти.

    Args:
        model_path: каталог dense или квантизированного checkpoint.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики и ответы модели, как в evaluate.
    """
    model_path = str(model_path)
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}
    quant_config = getattr(config, "quantization_config", None)
    if quant_config:
        # INT8 хранит тензор под именем weight. Сначала создаём CompressedLinear
        # с INT8-параметром без градиентов, затем слой сам распакует веса в forward.
        # Для упакованных W4A16 оставляем распаковку при загрузке.
        run_compressed = quant_config.get("format") in {"int-quantized", "naive-quantized"}
        quantization_kwargs["quantization_config"] = CompressedTensorsConfig(
            run_compressed=run_compressed,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map=device,
        **quantization_kwargs,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

def load_saved_predictions(run: str, ds: Dataset) -> list[str] | None:
    path = DEV_RESULTS / f"{run}_dev_predictions.jsonl"
    if not path.is_file():
        return None

    with path.open(encoding="utf-8") as stream:
        try:
            records = [json.loads(line) for line in stream]
        except json.JSONDecodeError:
            return None
    if len(records) != len(ds):
        return None

    predictions = []
    for row, record in zip(ds, records):
        for key in ("id", "text", "gold"):
            if row[key] != record.get(key):
                return None
        prediction = record.get("prediction")
        if not isinstance(prediction, str):
            return None
        predictions.append(prediction)
    return predictions


def save_dev_results(run: str, metrics: dict, predictions: list[str]) -> None:
    (DEV_RESULTS / f"{run}_dev.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8",
    )
    path = DEV_RESULTS / f"{run}_dev_predictions.jsonl"
    with path.open("w", encoding="utf-8") as stream:
        for row, prediction in zip(dev_data, predictions):
            record = {**row, "prediction": prediction}
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")

Посчитаем метрики бейзлайна.

In [ ]:
if RESUME_QUANTIZATION:
    baseline_predictions = load_saved_predictions("baseline", dev_data)
    if baseline_predictions is None:
        baseline_metrics, baseline_predictions = evaluate_checkpoint(BASELINE_PATH, dev_data)
    else:
        baseline_metrics = compute_metrics(baseline_predictions, dev_data["gold"], INTENTS, SLOT_TYPES)
else:
    baseline_metrics, baseline_predictions = evaluate(baseline_model, tokenizer, dev_data)
print(baseline_metrics)
save_dev_results("baseline", baseline_metrics, baseline_predictions)
results = [{
    "run": "baseline", "path": str(BASELINE_PATH),
    "method": "dense", "calibration": None, **baseline_metrics,
}]
del baseline_model
clean_memory()

## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [ ]:
### YOUR CODE HERE

import subprocess
import sys
import time
from tempfile import TemporaryDirectory

import dill
from compressed_tensors.quantization import QuantizationArgs, QuantizationScheme
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

CALIBRATION_SAMPLES = 64
CALIBRATION_MAX_LENGTH = 512
QUANT_IGNORE = ["lm_head"]

# Калибруем на исходных текстах обоих корпусов без эталонных ответов.
massive_calibration = train_data.shuffle(seed=SEED).select(range(CALIBRATION_SAMPLES))
wiki_train = load_dataset("Salesforce/wikitext", "wikitext-2-v1", split="train")
wiki_train = wiki_train.filter(lambda row: len(row["text"].split()) > 20)
wiki_calibration = wiki_train.shuffle(seed=SEED).select(range(CALIBRATION_SAMPLES))
calibration_data = {
    "massive": massive_calibration.select_columns(["text"]),
    "wikitext": wiki_calibration.select_columns(["text"]),
}

quant_runs = [
    ("gptq_massive", "GPTQ", "massive", 128, None),
    ("gptq_wikitext", "GPTQ", "wikitext", 128, None),
    ("gptq_g64_massive", "GPTQ", "massive", 64, None),
    ("awq_massive", "AWQ", "massive", 128, None),
    ("smoothquant_05_massive", "SmoothQuant", "massive", None, 0.5),
    ("smoothquant_08_massive", "SmoothQuant", "massive", None, 0.8),
]

def quant_recipe(method, group_size, smoothing_strength):
    if method == "SmoothQuant":
        return [
            SmoothQuantModifier(smoothing_strength=smoothing_strength),
            GPTQModifier(targets="Linear", scheme="W8A8", ignore=QUANT_IGNORE),
        ]

    weights = QuantizationArgs(
        num_bits=4, type="int", strategy="group", group_size=group_size,
        symmetric=True,
    )
    scheme = QuantizationScheme(targets=["Linear"], weights=weights)
    if method == "AWQ":
        return [AWQModifier(config_groups={"group_0": scheme}, ignore=QUANT_IGNORE)]
    return [GPTQModifier(config_groups={"group_0": scheme}, ignore=QUANT_IGNORE)]

def quantize_checkpoint(path, calibration, recipe, tokenizer, device):
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16, device_map=device,
    ).eval()
    try:
        oneshot(
            model=model, tokenizer=tokenizer, dataset=calibration, recipe=recipe,
            num_calibration_samples=len(calibration),
            max_seq_length=CALIBRATION_MAX_LENGTH,
        )
        model.save_pretrained(path, save_compressed=True)
        tokenizer.save_pretrained(path)
    finally:
        del model
        clean_memory()

def run_quantizations(assigned_runs, summaries_dir, device):
    worker_tokenizer = None
    if not RESUME_QUANTIZATION:
        worker_tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)
    with tqdm(assigned_runs, desc="Квантизация", unit="конфигурация") as progress:
        for run, method, corpus, group_size, smoothing in progress:
            path = ARTIFACTS / "quantized" / run
            calibration = calibration_data[corpus]
            predictions = None
            if RESUME_QUANTIZATION:
                predictions = load_saved_predictions(run, dev_data)
            else:
                progress.set_postfix_str(f"{run} · квантизация и сохранение")
                recipe = quant_recipe(method, group_size, smoothing)
                quantize_checkpoint(path, calibration, recipe, worker_tokenizer, device)

            if predictions is None:
                progress.set_postfix_str(f"{run} · оценка на dev")
                metrics, predictions = evaluate_checkpoint(path, dev_data, device=device)
                save_dev_results(run, metrics, predictions)
            else:
                progress.set_postfix_str(f"{run} · сохранённые ответы на dev")
                metrics = compute_metrics(predictions, dev_data["gold"], INTENTS, SLOT_TYPES)
            weight_mib = sum(file.stat().st_size for file in path.glob("*.safetensors")) / 2**20
            summary = {
                "run": run, "path": str(path), "method": method, "calibration": corpus,
                "scheme": "W8A8" if method == "SmoothQuant" else "W4A16",
                "calibration_samples": len(calibration), "group_size": group_size,
                "symmetric_weights": True, "smoothing_strength": smoothing,
                "weight_mib": weight_mib, **metrics,
            }
            (summaries_dir / f"{run}.json").write_text(
                json.dumps(summary), encoding="utf-8",
            )

def run_on_two_gpus(summaries_dir):
    # Сохраняем порядок GPU, уже заданный окружением.
    visible_gpus = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
    logs_dir = ARTIFACTS / "quantization_logs"
    logs_dir.mkdir(exist_ok=True)
    workers = []

    # Передаём функции из ноутбука в отдельный Python-процесс.
    worker_code = """
import sys
import dill

with open(sys.argv[1], "rb") as stream:
    run, args = dill.load(stream)
run(*args)
"""
    try:
        for index in range(2):
            assigned_runs = quant_runs[index::2]
            job_path = summaries_dir / f"gpu_{index}.pkl"
            with job_path.open("wb") as stream:
                args = (assigned_runs, summaries_dir, "cuda:0")
                dill.dump((run_quantizations, args), stream, recurse=True)

            # После ограничения видимости каждая карта внутри своего процесса — cuda:0.
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = visible_gpus[index].strip()
            env["TOKENIZERS_PARALLELISM"] = "false"
            log_name = f"gpu_{index}_resume.log" if RESUME_QUANTIZATION else f"gpu_{index}.log"
            log_path = logs_dir / log_name
            names = ", ".join(run[0] for run in assigned_runs)
            print(f"GPU {index}: {names}. Лог: {log_path}")
            with log_path.open("w", encoding="utf-8") as log:
                process = subprocess.Popen(
                    [sys.executable, "-u", "-c", worker_code, str(job_path)],
                    env=env, stdout=log, stderr=subprocess.STDOUT,
                )
            workers.append((process, log_path))

        with tqdm(total=len(quant_runs), desc="Квантизация на двух GPU", unit="конфигурация") as progress:
            while any(process.poll() is None for process, _ in workers):
                done = len(list(summaries_dir.glob("*.json")))
                progress.update(done - progress.n)
                time.sleep(1)
            done = len(list(summaries_dir.glob("*.json")))
            progress.update(done - progress.n)

        failed_logs = [str(log) for process, log in workers if process.returncode != 0]
        if failed_logs:
            raise RuntimeError("Ошибка в процессе квантизации. Логи: " + ", ".join(failed_logs))
    finally:
        # При остановке ячейки завершаем оба дочерних процесса.
        for process, _ in workers:
            if process.poll() is None:
                process.terminate()
        for process, _ in workers:
            process.wait()

# Проверяем пути до запуска процессов. При возобновлении нужны все чекпоинты.
for run, *_ in quant_runs:
    path = ARTIFACTS / "quantized" / run
    if RESUME_QUANTIZATION:
        require_saved_checkpoint(path)
    else:
        ensure_empty_dir(path)

with TemporaryDirectory(prefix="massive_quantization_") as temp:
    summaries_dir = Path(temp)
    if torch.cuda.device_count() >= 2:
        run_on_two_gpus(summaries_dir)
    else:
        run_quantizations(quant_runs, summaries_dir, DEVICE)

    for run, *_ in quant_runs:
        summary = (summaries_dir / f"{run}.json").read_text(encoding="utf-8")
        results.append(json.loads(summary))

quant_results = pd.DataFrame(results)
quant_results["delta_quality_vs_baseline"] = quant_results["quality"] - baseline_metrics["quality"]
quant_results.to_csv(ARTIFACTS / "quantization_results.csv", index=False)
display(quant_results.round(4))

quality_by_run = quant_results.set_index("run")["quality"]
comparisons = [
    ("GPTQ: MASSIVE минус WikiText", "gptq_massive", "gptq_wikitext"),
    ("GPTQ: группа 64 минус 128", "gptq_g64_massive", "gptq_massive"),
    ("SmoothQuant: alpha 0.8 минус 0.5", "smoothquant_08_massive", "smoothquant_05_massive"),
]
for label, first, second in comparisons:
    if first not in quality_by_run.index or second not in quality_by_run.index:
        continue
    delta = quality_by_run[first] - quality_by_run[second]
    print(f"{label} → {delta * 100:+.2f} п.п. Quality")

### Выводы по квантизации

- **MASSIVE train и WikiText.** При GPTQ W4A16 с группой 128 Quality составил 0,16537 на MASSIVE и 0,16543 на WikiText. Разница — 0,0065 п.п. в пользу WikiText. При калибровке на MASSIVE выше Slot Micro F1 (0,1003 против 0,0809) и ниже Intent Accuracy (0,2305 против 0,2500). На этой подвыборке выигрыш одного корпуса по Quality практически нулевой.
- **Размер группы.** У GPTQ на MASSIVE уменьшение группы со 128 до 64 подняло Quality с 0,1654 до 0,2884 (+12,30 п.п.). Это крупнейшее изменение среди проверенных настроек. Суммарный размер шардов весов вырос примерно на 104 MiB. GPTQ с группой 64 дал лучший Quality среди квантизированных моделей, на 2,45 п.п. выше бейзлайна. Frame Exact Match при этом ниже, чем у бейзлайна (0,0977 против 0,1094).
- **Метод и сглаживание.** При одинаковом W4A16, группе 128 и калибровке на MASSIVE AWQ дал 0,2163 против 0,1654 у GPTQ (+5,09 п.п.). В связке SmoothQuant + GPTQ W8A8 увеличение коэффициента сглаживания с 0,5 до 0,8 снизило Quality с 0,2717 до 0,2625 (−0,91 п.п.). Сравнение W4A16 и W8A8 одновременно меняет схему и рецепт, так что эффект одной битности из этих запусков не выделяется.

Оценка проведена на одной подвыборке из 256 запросов. В исходном прогоне Transformers подставил параметры sampling из конфигурации Qwen3. Приведённые разницы относятся к сохранённым предсказаниям.


## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [ ]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if (
    tokenizer.get_vocab() != teacher_tokenizer.get_vocab()
    or tokenizer.chat_template != teacher_tokenizer.chat_template
    or tokenizer.eos_token_id != teacher_tokenizer.eos_token_id
    or tokenizer.pad_token_id != teacher_tokenizer.pad_token_id
):
    raise ValueError("Токенизаторы ученика и учителя различаются.")
del teacher_tokenizer

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
student_vocab_size = AutoConfig.from_pretrained(STUDENT_MODEL_ID).vocab_size
if teacher.config.vocab_size != student_vocab_size:
    raise ValueError("Размерности выходных логитов teacher и student различаются.")

In [ ]:
### YOUR CODE HERE

teacher_metrics, teacher_predictions = evaluate(teacher, tokenizer, dev_data)
save_dev_results("teacher", teacher_metrics, teacher_predictions)

teacher_comparison = pd.DataFrame([
    {"model": BASE_MODEL_ID, **baseline_metrics},
    {"model": TEACHER_MODEL_ID, **teacher_metrics},
])
display(teacher_comparison.round(4))
teacher_delta = teacher_metrics["quality"] - baseline_metrics["quality"]
print(f"Разница между учителем и бейзлайном по Quality: {teacher_delta * 100:+.2f} п.п.")

### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [ ]:
DISTILL_SAMPLES = 1024

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = ensure_empty_dir(output_path)
    records = []
    row_list = list(rows)

    ### YOUR CODE HERE
    
    model.eval()
    for start in tqdm(range(0, len(row_list), GENERATE_BATCH_SIZE), desc="Ответы учителя"):
        batch = row_list[start:start + GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, row["text"]) for row in batch]
        answers = generate_answer_ids(model, tokenizer, prompts)
        texts = tokenizer.batch_decode(answers, skip_special_tokens=True)
        for row, prompt_ids, answer_ids, text in zip(batch, prompts, answers, texts):
            records.append({
                **row, "teacher_answer": text,
                "prompt_ids": prompt_ids, "answer_ids": answer_ids,
            })

    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [ ]:
distill_rows = train_data.shuffle(seed=SEED).select(
    range(min(DISTILL_SAMPLES, len(train_data)))
)
TEACHER_ANSWERS = collect_teacher_answers(
    teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
)
del teacher
clean_memory()

### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [ ]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path, max_rows: int) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
            max_rows: сколько ответов прочитать до фильтрации по схеме JSON.
        """
        self.rows = [] # Список примеров из файла

        ### YOUR CODE HERE
        ### Fill rows

        with (Path(path) / "answers.jsonl").open(encoding="utf-8") as stream:
            for index, line in enumerate(stream):
                if index >= max_rows:
                    break
                row = json.loads(line)
                answer = parse_answer(row["teacher_answer"], INTENTS, SLOT_TYPES)
                if row["answer_ids"] and answer is not None:
                    self.rows.append(row)

    def __len__(self) -> int:
        """
        Число примеров с валидным ответом учителя.
        """
        ### YOUR CODE HERE
        
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        # YOUR CODE HERE
        input_ids = row["prompt_ids"] + row["answer_ids"]
        # YOUR CODE HERE
        labels = [-100] * len(row["prompt_ids"]) + row["answer_ids"]
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [ ]:
def train_student(run, max_rows, learning_rate, epochs):
    set_seed(SEED)
    train_dataset = TeacherAnswerDataset(TEACHER_ANSWERS, max_rows=max_rows)
    if len(train_dataset) == 0:
        raise ValueError("В выбранной части train нет валидных ответов учителя.")
    print(f"{run}: {len(train_dataset)} валидных ответов из {max_rows} запрошенных")

    path = ensure_empty_dir(ARTIFACTS / "student_trials" / run)
    student = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16,
    )
    student.config.use_cache = False

    args = TrainingArguments(
        output_dir=str(path),
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        optim="adafactor",
        bf16=torch.cuda.is_available(),
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )
    sft = Trainer(
        model=student,
        args=args,
        train_dataset=train_dataset,
        data_collator=collate_teacher_answers,
    )
    sft.train()
    student.config.use_cache = True
    sft.save_model(str(path))
    tokenizer.save_pretrained(path)
    del sft, student
    clean_memory()

    metrics, predictions = evaluate_checkpoint(path, dev_data)
    save_dev_results(run, metrics, predictions)
    return {
        "run": run, "path": str(path), "train_requested": max_rows,
        "train_valid": len(train_dataset), "learning_rate": learning_rate,
        "epochs": epochs, **metrics,
    }

In [ ]:
student_results = [
    train_student("student_128_lr1e-5", max_rows=128, learning_rate=1e-5, epochs=2),
]
display(pd.DataFrame(student_results).round(4))

### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [ ]:
### YOUR CODE HERE

student_configs = [
    ("student_large_lr1e-5", DISTILL_SAMPLES, 1e-5, 2),
]
# Третий прогон требует места для ещё одной копии Qwen3-8B.
RUN_LR_ABLATION = False
if RUN_LR_ABLATION:
    student_configs.append(("student_large_lr5e-6", DISTILL_SAMPLES, 5e-6, 2))
for run, max_rows, learning_rate, epochs in student_configs:
    student_results.append(train_student(run, max_rows, learning_rate, epochs))

# Выбираем модель по Quality на общей dev-выборке.
best_student = max(student_results, key=lambda result: result["quality"])
best_run = best_student["run"]
student_path = ARTIFACTS / "student_sft"
Path(best_student["path"]).rename(student_path)
best_student["path"] = str(student_path)

for suffix in ["_dev.json", "_dev_predictions.jsonl"]:
    shutil.copyfile(DEV_RESULTS / f"{best_run}{suffix}", DEV_RESULTS / f"student_sft{suffix}")

teacher_quality = teacher_metrics["quality"]
target_quality = 0.9 * teacher_quality
distillation_results = pd.DataFrame([
    {"run": "baseline", "path": str(BASELINE_PATH), **baseline_metrics},
    {"run": "teacher", "path": TEACHER_MODEL_ID, **teacher_metrics},
    *student_results,
])
distillation_results["delta_quality_vs_baseline"] = (
    distillation_results["quality"] - baseline_metrics["quality"]
)
distillation_results["delta_quality_vs_teacher"] = (
    distillation_results["quality"] - teacher_quality
)
distillation_results["selected_student"] = distillation_results["run"] == best_run
distillation_results.to_csv(ARTIFACTS / "distillation_results.csv", index=False)
display(distillation_results.round(4))

best_quality = best_student["quality"]
print(f"Лучший ученик: {best_run}, Quality = {best_quality:.4f}")
print(f"Цель (90% Quality учителя): {target_quality:.4f}")
print("Цель достигнута:", best_quality >= target_quality)
print(f"Разница с бейзлайном: {(best_quality - baseline_metrics['quality']) * 100:+.2f} п.п.")
print("Ученик превосходит бейзлайн:", best_quality > baseline_metrics["quality"])

### Выводы по hard-label дистилляции

- Учитель Qwen3-32B получил **0,5598 Quality**, на **29,59 п.п.** выше Qwen3-8B (0,2639). Из 1 024 сгенерированных ответов на train фильтр по схеме пропустил 1 003. Для малого прогона прошли 126 из первых 128 ответов.
- При двух эпохах и `lr=1e-5` переход с 126 на 1 003 валидных примера поднял Quality ученика с **0,5170 до 0,5916** (+7,45 п.п.). Лучший ученик набрал **105,68% Quality учителя** при требовании минимум 90% (0,5038). Он прибавил **32,77 п.п.** к бейзлайну и **3,18 п.п.** к учителю. Frame Exact Match у ученика 0,3008, у учителя 0,3047.
- Прогон с 1 003 примерами и `lr=5e-6` завершил обучение, но сохранение весов прервалось из-за нехватки места. Оценки на dev для него нет, поэтому влияние learning rate осталось открытым.

Целевые ответы в исходном прогоне — последовательности токенов учителя, сгенерированные с sampling. В ноутбуке для нового запуска явно задан greedy decoding.
